In [3]:
"""Tutorial: Complete Bayesian Optimal Experimental Design workflow.

This is the main entry point demonstrating the full pyCBOED workflow:
1. Setting up a forward PDE model
2. Defining a Gaussian Process prior
3. Generating synthetic observations
4. Performing Bayesian inference
5. Evaluating design criteria
6. Visualizing results

This example uses a 1D advection-diffusion PDE with a Gaussian Process prior
on the initial condition. It demonstrates linear Gaussian inference and
multiple design optimality criteria.
"""
import numpy as np
from boed.core.base import validate_positive_definite
from boed.core import make_u0
from boed.priors.kernels import Gaussian, Matern12, Matern32
from boed.priors.gp_priors import GaussianProcessPrior
from boed.core.noise import NoiseModel
from boed.pde.advection_diffusion import AdvectionDiffusion1D_CN
from boed.viz.boed_visualizer_pro import BOEDVisualizerPro
from boed.inference import LinearGaussianModel
from boed.observations.sensors import SpaceTimeSensors
# from boed.core.utils import compute_W,generate_synthetic_observations
from boed.design.greedy import run_greedy_oed, run_sboed_trajectories, run_sequential_step_oed

SEED = 42
np.random.seed(SEED)


In [4]:
# ===============================================================================
# PARAMETERS
# ===============================================================================
N = 150
dt = 0.005
n_steps = 10
diffusivity = 0.01
velocity = 0.5


In [5]:
# ------------------- PDE -------------------
model = AdvectionDiffusion1D_CN(N, dt, diffusivity=diffusivity, velocity=velocity)
x_grid = np.linspace(0,1,N)
stable, msg = model.check_stability()
print(msg)

CFL=0.378, stable=True


In [7]:
# ------------------- Prior -------------------
kernel = Gaussian(length_scale=1, sigma=1.0)
prior_process = GaussianProcessPrior(kernel, nx=N, mu=None)
Sigma_prior = prior_process.Sigma
mu_prior = prior_process.mu
x_prior = prior_process.x
validate_positive_definite(Sigma_prior, "Prior covariance")

In [8]:
# ------------------- Synthetic data -------------------
u_true = make_u0(
    x_grid,
    "double_gaussian",
    centers=(0.3, 0.7),
    widths=(0.5, 0.1),
    amplitudes=(1.0, 0.5),
)
trajectory = model.evolve(u_true, n_steps)

In [9]:
selected_design, history, current_Sigma = run_greedy_oed(
    model,
    Sigma_prior,
    noise_model = 0.001,
    candidates_x = np.arange(0,N,1),
    candidates_t =  np.linspace(0, n_steps, 10, dtype=int),
    n_budget = 10,
    criterion_type="D",
)

--- Greedy OED optimization (criterion D) ---


KeyboardInterrupt: 

In [ ]:
selected_design, history, current_Sigma= run_sequential_step_oed(
    model,
    Sigma_prior,
    noise_model = 0.001,
    candidates_x = np.arange(0,N,1),
    times_to_observe =  np.linspace(0, n_steps, 10, dtype=int),
    budget_per_step = 2,
    criterion_type="A",
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_sensor_trajectories(selected_design, n_spatial_points, x_grid=None, title=None):
    """
    Affiche l'évolution temporelle des positions des capteurs.
    Gère automatiquement le cas d'un capteur unique ou de multiples capteurs.
    """
    design_array = np.array(selected_design)
    # Tri par temps (colonne 1) pour assurer la continuité des lignes
    design_array = design_array[design_array[:, 1].argsort()]
    
    t_vals = design_array[:, 1]
    x_indices = design_array[:, 0]
    
    # Conversion indices -> positions physiques si x_grid est fourni
    y_vals = x_grid[x_indices] if x_grid is not None else x_indices
    y_label = "Position physique (x)" if x_grid is not None else "Position (indice x)"
    
    # Détermination du nombre de capteurs par pas de temps
    unique_times = np.unique(t_vals)
    n_per_step = np.sum(t_vals == unique_times[0])
    
    plt.figure(figsize=(12, 7))
    
    # 1. Tracé des trajectoires (lignes de liaison)
    # On relie le i-ème capteur de chaque pas de temps entre eux
    for i in range(n_per_step):
        path_t = unique_times
        path_y = y_vals[i::n_per_step]
        plt.plot(path_t, path_y, color='gray', linestyle='--', alpha=0.4, lw=1.5, zorder=1)

    # 2. Tracé des points avec dégradé de couleur temporel
    scatter = plt.scatter(t_vals, y_vals, c=t_vals, cmap='plasma', 
                          s=80, edgecolors='black', linewidths=0.5, zorder=2)
    
    # 3. Cosmétique
    plt.title(title or f"Trajectoires SBOED ({n_per_step} capteur(s) par pas)")
    plt.xlabel("Temps (indices t)")
    plt.ylabel(y_label)
    
    if x_grid is None:
        plt.ylim(-1, n_spatial_points)
    else:
        plt.ylim(x_grid.min(), x_grid.max())

    plt.grid(True, alpha=0.2, ls=':')
    plt.colorbar(scatter, label="Évolution temporelle")
    plt.tight_layout()
    plt.show()

In [ ]:
plot_sensor_trajectories(selected_design=selected_design, n_spatial_points=N,x_grid=x_grid)

In [ ]:
trajectory_history, current_Sigma= run_sboed_trajectories(
    model,
    Sigma_prior,
    noise_model = 0.001,
    candidates_x = np.arange(0,N,1),
    times =  np.linspace(0, n_steps, 10, dtype=int),
    budget_per_step = 6
)

In [ ]:
def plot_sensor_flow(trajectory_data, n_x):
    t_vals = trajectory_data[:, 0]
    x_vals = trajectory_data[:, 1]
    
    plt.figure(figsize=(12, 6))
    
    # On trace les points individuels
    plt.scatter(t_vals, x_vals, c=t_vals, cmap='plasma', edgecolors='k', s=60, label='Capteurs sélectionnés')
    
    # On relie les points par "index de capteur" pour simuler des trajectoires
    # Si budget_per_step = 2, on relie le 1er du temps t au 1er du temps t+1
    unique_times = np.unique(t_vals)
    n_per_step = np.sum(t_vals == unique_times[0])
    
    for i in range(n_per_step):
        path_x = x_vals[i::n_per_step]
        path_t = t_vals[i::n_per_step]
        plt.plot(path_t, path_x, alpha=0.3, color='gray', linestyle='--')

    plt.title("Trajectoires adaptatives des capteurs (SBOED)")
    plt.xlabel("Temps (t)")
    plt.ylabel("Position spatiale (x)")
    plt.ylim(0, n_x)
    plt.grid(True, alpha=0.2)
    plt.show()

In [ ]:
plot_sensor_flow(trajectory_data=trajectory_history,n_x=N)

In [ ]:
plot_sensor_trajectories(selected_design=selected_design, n_spatial_points=N,x_grid=x_grid)